<div dir="rtl">

# מטלת איסוף נתוני סרטים
### זיו נגד 322558271
### תפארת בלוקה 325204113
### GitHub: https://github.com/zivnagad/project_movies_c

In [1]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup

In [2]:
df_titles = pd.read_csv("title.basics.tsv.gz", sep="\t", compression="gzip", na_values="\\N",low_memory=False)
df_titles.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894.0,NaN,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892.0,NaN,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892.0,NaN,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,Un bon bock,0,1892.0,NaN,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893.0,NaN,1,Short


In [3]:
ratings = pd.read_csv("title.ratings.tsv.gz", sep="\t", compression="gzip", na_values="\\N")
ratings.head()

,tconst,averageRating,numVotes
0,tt0000001,5.7,2213
1,tt0000002,5.5,317
2,tt0000003,6.4,2327
3,tt0000004,5.1,199
4,tt0000005,6.2,3047


In [4]:
principals = pd.read_csv("title.principals.tsv.gz", sep="\t", compression="gzip", na_values="\\N", low_memory=False)
principals.head()

,tconst,ordering,nconst,category,job,characters
0,tt0000001,1,nm1588970,self,NaN,"[""Self""]"
1,tt0000001,2,nm0005690,director,NaN,NaN
2,tt0000001,3,nm0005690,producer,producer,NaN
3,tt0000001,4,nm0374658,cinematographer,director of photography,NaN
4,tt0000002,1,nm0721526,director,NaN,NaN


In [9]:
df_name = pd.read_csv("name.basics.tsv.gz", sep="\t", na_values="\\N", low_memory=False, compression="gzip")
df_name.head()

,nconst,primaryName,birthYear,deathYear,primaryProfession,knownForTitles
0,nm0000001,Fred Astaire,1899.0,1987.0,"actor,miscellaneous,producer","tt0072308,tt0050419,tt0027125,tt0025164"
1,nm0000002,Lauren Bacall,1924.0,2014.0,"actress,miscellaneous,soundtrack","tt0037382,tt0075213,tt0038355,tt0045891"
2,nm0000003,Brigitte Bardot,1934.0,2025.0,"actress,music_department,producer","tt0057345,tt0049189,tt0056404,tt0054452"
3,nm0000004,John Belushi,1949.0,1982.0,"actor,writer,music_department","tt0072562,tt0077975,tt0080455,tt0078723"
4,nm0000005,Ingmar Bergman,1918.0,2007.0,"writer,director,actor","tt0050986,tt0069467,tt0050976,tt0083922"


<div dir="rtl">

#### סינון "titleType" לסרטים בלבד
#### סינון שם הסרט שיתחיל באות C בלבד
#### שנת יציאת הסרט עד שנת 2024
#### אורך הסרט בין 60 ל300 דקות

#### המרת שנת הסרט ל"int" כי אנחנו לא בטוחים שהוא מוגדר כך (אולי הוא object) ו"Int64" כדי שידע להכיל ערכים חסרים:

In [12]:
# סינון רק לסרטים מסוג movie, עד 2024, עם זמן ריצה תקין, ושם שמתחיל באות C
df_movies_c = df_titles[
    (df_titles["titleType"] == "movie") &
    (df_titles["primaryTitle"].str.startswith("C", na=False)) &
    (pd.to_numeric(df_titles["startYear"], errors="coerce") <= 2024) &
    (pd.to_numeric(df_titles["runtimeMinutes"], errors="coerce").between(60, 300))
].copy()

# המרת עמודות מספריות
#לבדוק אם צריך את זה, כי כתוב בדף של ענת שזה אינט
df_movies_c["startYear"] = pd.to_numeric(df_movies_c["startYear"], errors="coerce").astype("Int64")
df_movies_c["runtimeMinutes"] = pd.to_numeric(df_movies_c["runtimeMinutes"], errors="coerce").astype("Int64")

df_movies_c.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
2076,tt0002101,movie,Cleopatra,Cleopatra,0,1912,NaN,100,"Drama,History"
3278,tt0003309,movie,Captive Souls,Rablélek,0,1914,NaN,99,NaN
3704,tt0003740,movie,Cabiria,Cabiria,0,1914,NaN,148,"Adventure,Drama,History"
3712,tt0003748,movie,Captain Alvarez,Captain Alvarez,0,1914,NaN,60,Drama
4997,tt0005061,movie,Carmen,Carmen,0,1915,NaN,60,Drama


<div dir="rtl">

#### נעשה חיבור טבלת df_titles לטבלת df_ratings לפי מזהה הסרט ("tconst"):

In [17]:
df_movies_c = df_movies_c.merge(
    ratings,
    on="tconst",
    how="left"
)

df_movies_c.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes
0,tt0002101,movie,Cleopatra,Cleopatra,0,1912,NaN,100,"Drama,History",5.1,673.0
1,tt0003309,movie,Captive Souls,Rablélek,0,1914,NaN,99,NaN,4.3,39.0
2,tt0003740,movie,Cabiria,Cabiria,0,1914,NaN,148,"Adventure,Drama,History",7.1,4338.0
3,tt0003748,movie,Captain Alvarez,Captain Alvarez,0,1914,NaN,60,Drama,8.0,22.0
4,tt0005061,movie,Carmen,Carmen,0,1915,NaN,60,Drama,4.4,55.0


<div dir="rtl">

#### נכין את 5 השחקנים המובילים בכל סרט מטבלת df_principals:

In [97]:
# רק שחקנים
actors = principals[principals["category"] == "actor"].copy()

# רק סרטים שנמצאים בדאטה שלנו
actors_c = actors[actors["tconst"].isin(df_movies_c["tconst"])].copy()

# מיון לפי ordering כדי לקחת את הראשיים
actors_c = actors_c.sort_values(["tconst", "ordering"])

# יצירת רשימה של עד 5 שחקנים לכל סרט
lead_actors = (
    actors_c
    .groupby("tconst")["nconst"]
    .apply(lambda x: list(x.head(5)))
    .reset_index()
    .rename(columns={"nconst": "lead_actors_ids"})
)

lead_actors.head()

,tconst,lead_actors_ids
0,tt0002101,"[nm1950505, nm0397513, nm0906610, nm0651716, n..."
1,tt0003740,"[nm0656034, nm0856422, nm0610618, nm0224122, n..."
2,tt0006517,"[nm0382730, nm0121706, nm0279508, nm0294058, n..."
3,tt0007801,"[nm0500021, nm0356159, nm0741182, nm0562197, n..."
4,tt0008950,"[nm0509573, nm0081510, nm0175761, nm0472174, n..."


<div dir="rtl">

#### ונחבר לדאטה הסופית:

In [99]:
df_final = df_movies_c.merge(
    lead_actors,
    on="tconst",
    how="left"
)

df_final.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes,lead_actors_ids
0,tt0317248,movie,City of God,Cidade de Deus,0,2002,NaN,130,"Crime,Drama",8.6,870147.0,"[nm1179105, nm1129884, nm0618690, nm1249574, n..."
1,tt0034583,movie,Casablanca,Casablanca,0,1942,NaN,102,"Drama,Romance,War",8.5,655268.0,"[nm0000007, nm0002134, nm0001647, nm0891998, n..."
2,tt0264464,movie,Catch Me If You Can,Catch Me If You Can,0,2002,NaN,141,"Biography,Crime,Drama",8.1,1236176.0,"[nm0000138, nm0000158, nm0000686, nm0000640, n..."
3,tt2380307,movie,Coco,Coco,0,2017,NaN,105,"Adventure,Animation,Drama",8.4,705833.0,"[nm5645519, nm0305558, nm0000973, nm0131781, n..."
4,tt0112641,movie,Casino,Casino,0,1995,NaN,178,"Crime,Drama",8.2,618855.0,"[nm0000134, nm0000582, nm0000249, nm0725543, n..."


<div dir="rtl">

#### נבדוק את מספר השורות והעמודות ונשאיר רק את הרלוונטיות:

In [29]:
df_final.shape

(18340, 12)

In [31]:
df_final = df_final[
    [
        "tconst",
        "primaryTitle",
        "startYear",
        "genres",
        "lead_actors_ids",
        "runtimeMinutes",
        "averageRating",
        "numVotes"
    ]
]

df_final.head()

,tconst,primaryTitle,startYear,genres,lead_actors_ids,runtimeMinutes,averageRating,numVotes
0,tt0002101,Cleopatra,1912,"Drama,History","[nm1950505, nm0397513, nm0906610, nm0651716, n...",100,5.1,673.0
1,tt0003309,Captive Souls,1914,NaN,"[nm0753583, nm0223460, nm0862278, nm9892419, n...",99,4.3,39.0
2,tt0003740,Cabiria,1914,"Adventure,Drama,History","[nm0656034, nm0856422, nm0610618, nm0224122, n...",148,7.1,4338.0
3,tt0003748,Captain Alvarez,1914,Drama,"[nm0853336, nm0822481, nm0392429, nm0496470, n...",60,8.0,22.0
4,tt0005061,Carmen,1915,Drama,"[nm0511598, nm0361882, nm0546121, nm0212040, n...",60,4.4,55.0


<div dir="rtl">

#### נגדיר את הסרטים לפי דירוג + פופולריות מהגבוה לנמוך (מהסרט החזק ביותר להכי פחות):

In [52]:
df_movies_c["success_score"] = df_movies_c["averageRating"] * np.log1p(df_movies_c["numVotes"])

In [54]:
df_movies_c = df_movies_c.sort_values(
    by="success_score",
    ascending=False
).head(5000).reset_index(drop=True)

df_movies_c = df_movies_c.drop(columns=["success_score"])


df_movies_c.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes
0,tt0317248,movie,City of God,Cidade de Deus,0,2002,NaN,130,"Crime,Drama",8.6,870147.0
1,tt0034583,movie,Casablanca,Casablanca,0,1942,NaN,102,"Drama,Romance,War",8.5,655268.0
2,tt0264464,movie,Catch Me If You Can,Catch Me If You Can,0,2002,NaN,141,"Biography,Crime,Drama",8.1,1236176.0
3,tt2380307,movie,Coco,Coco,0,2017,NaN,105,"Adventure,Animation,Drama",8.4,705833.0
4,tt0112641,movie,Casino,Casino,0,1995,NaN,178,"Crime,Drama",8.2,618855.0


<div dir="rtl">

#### נחבר לדאטה הסופי:

In [94]:
df_final = df_movies_c.merge(
    lead_actors,
    on="tconst",
    how="left"
)

df_final.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes,lead_actors_ids
0,tt0317248,movie,City of God,Cidade de Deus,0,2002,NaN,130,"Crime,Drama",8.6,870147.0,"[nm1179105, nm1129884, nm0618690, nm1249574, n..."
1,tt0034583,movie,Casablanca,Casablanca,0,1942,NaN,102,"Drama,Romance,War",8.5,655268.0,"[nm0000007, nm0002134, nm0001647, nm0891998, n..."
2,tt0264464,movie,Catch Me If You Can,Catch Me If You Can,0,2002,NaN,141,"Biography,Crime,Drama",8.1,1236176.0,"[nm0000138, nm0000158, nm0000686, nm0000640, n..."
3,tt2380307,movie,Coco,Coco,0,2017,NaN,105,"Adventure,Animation,Drama",8.4,705833.0,"[nm5645519, nm0305558, nm0000973, nm0131781, n..."
4,tt0112641,movie,Casino,Casino,0,1995,NaN,178,"Crime,Drama",8.2,618855.0,"[nm0000134, nm0000582, nm0000249, nm0725543, n..."


<div dir="rtl">

#### עכשיו נעבור לחלק של ויקיפדיה:

In [102]:
import time
import re;

<div dir="rtl">

#### נכתוב פונקציה שמחפשת עמוד Wikipedia לפי שם הסרט והשנה שבה יצא:

In [105]:
def search_wikipedia_page(title, year=None):
    url = "https://en.wikipedia.org/w/api.php"

    if pd.notna(year):
        search_query = f'{title} {int(year)} film'
    else:
        search_query = f'{title} film'

    params = {
        "action": "query",
        "list": "search",
        "srsearch": search_query,
        "format": "json",
        "srlimit": 5
    }

    headers = {
        "User-Agent": "MovieDataProject/1.0 (student project)"
    }

    response = requests.get(url, params=params, headers=headers)

    print(response.status_code)

    if response.status_code == 429:
        print("Too many requests - waiting...")
        time.sleep(10)
        return None

    if response.status_code != 200:
        return None

    try:
        data = response.json()
    except:
        return None

    results = data.get("query", {}).get("search", [])

    for result in results:
        page_title = result["title"]
    
        if "disambiguation" in page_title.lower():
            continue
    
        if title.lower() not in page_title.lower():
            continue
    
        return page_title

    
    #for result in results:
        #page_title = result["title"]

     #   if "disambiguation" in page_title.lower():
         #   continue

      #  if "film" in page_title.lower() or str(int(year)) in page_title:
        #    return page_title

    if len(results) > 0:
        return results[0]["title"]

    return None

<div dir="rtl">

#### נעשה בדיקה על סרט אקראי:

In [107]:
page = search_wikipedia_page("Contact", 1997)
print(page)

200
Contact (1997 American film)


<div dir="rtl">

#### נעשה פונקציה ששולפת מידע מטבלת הסרט בויקיפדיה:

In [110]:
def get_wikipedia_movie_info(page_title):
    if page_title is None:
        return {
            "Language": None,
            "Country": None,
            "budget": None,
            "BoxOffice": None
        }

    url_title = page_title.replace(" ", "_")
    url = f"https://en.wikipedia.org/wiki/{url_title}"

    headers = {
        "User-Agent": "MovieDataProject/1.0 (student project)"
    }

    response = requests.get(url, headers=headers)

    if response.status_code == 429:
        print("Too many requests - waiting...")
        time.sleep(10)
        return {
            "Language": None,
            "Country": None,
            "budget": None,
            "BoxOffice": None
        }

    if response.status_code != 200:
        return {
            "Language": None,
            "Country": None,
            "budget": None,
            "BoxOffice": None
        }

    soup = BeautifulSoup(response.text, "html.parser")

    infobox = soup.find("table", class_=lambda x: x and "infobox" in x)

    if infobox is None:
        print("No infobox found")
        return {
            "Language": None,
            "Country": None,
            "budget": None,
            "BoxOffice": None
        }

    info = {
        "Language": None,
        "Country": None,
        "budget": None,
        "BoxOffice": None
    }

    for row in infobox.find_all("tr"):
        header = row.find("th")
        value = row.find("td")

        if header is None or value is None:
            continue

        key = header.get_text(" ", strip=True).lower()

        for sup in value.find_all("sup"):
            sup.decompose()
        
        val = value.get_text(separator=", ", strip=True) #כדי לשים פסיק בין המדינות כשזה יותר ממדינה אחת

        val = re.sub(r"\[\s*\d+\s*\]", "", val)
        val = " ".join(val.split())
         #כדי להוריד את המספרים שמעל המדינה בויקיפדיה(כמו בסרט הראשון בדאטה שלנו)

        if "language" in key:
            info["Language"] = val

        elif "countr" in key: #כי לפעמים יש מדינה ברבים ולפעמים ביחיד אז שיתפוס לשניהם
            info["Country"] = val 

        elif "budget" in key:
            info["budget"] = val

        elif "box office" in key:
            info["BoxOffice"] = val

    return info

In [112]:
#בדיקה על סרט מסוים
page = search_wikipedia_page("Contact", 1997)
info = get_wikipedia_movie_info(page)
print(info)

200
{'Language': 'English', 'Country': 'United States', 'budget': '$90 million', 'BoxOffice': '$171 million'}


<div dir="rtl">

#### נריץ על 5 סרטים לבדיקה:

In [223]:
wiki_results = []

for index, row in df_final.head(5).iterrows():

    title = row["primaryTitle"]
    year = row["startYear"]

    print(f"Processing: {title} ({year})")

    page_title = search_wikipedia_page(title, year)

    print("Wikipedia page:", page_title)

    info = get_wikipedia_movie_info(page_title)

    info["tconst"] = row["tconst"]
    info["primaryTitle"] = title

    wiki_results.append(info)

    time.sleep(3)

Processing: City of God (2002)
200
Wikipedia page: City of God (2002 film)
Processing: Casablanca (1942)
200
Wikipedia page: Casablanca (film)
Processing: Catch Me If You Can (2002)
200
Wikipedia page: Catch Me If You Can
Processing: Coco (2017)
200
Wikipedia page: Coco (2017 film)
Processing: Casino (1995)
200
Wikipedia page: Casino (1995 film)


In [226]:
df_wiki_test = pd.DataFrame(wiki_results)

df_wiki_test

,Language,Country,budget,BoxOffice,tconst,primaryTitle
0,Portuguese,"Brazil, France, Germany, United States",$3.3 million,$30.6 million,tt0317248,City of God
1,English,United States,"$878,000, –$1 million","$3.7, –6.9 million",tt0034583,Casablanca
2,English,United States,$52 million,$352.1 million,tt0264464,Catch Me If You Can
3,English,United States,"$175–225, million","$814.3, million",tt2380307,Coco
4,English,"United States, France",$40–50 million,$116.1 million,tt0112641,Casino


<div dir="rtl">

#### נוסיף עמודת plot דרך OMDb:

In [115]:
API_KEY = "70760fbc"

def get_plot_from_omdb(title, year):
    url = f"http://www.omdbapi.com/?t={title}&y={int(year)}&apikey={API_KEY}"
    try:
        r = requests.get(url, timeout=10)
        data = r.json()
        if data.get("Response") == "False":
            return None
        return data.get("Plot") or None
    except:
        return None

<div dir="rtl">

#### נריץ את הפונקציה על כל הדאטה שלנו

In [124]:
df_final["Language"] = None
df_final["Country"] = None
df_final["budget"] = None
df_final["BoxOffice"] = None
df_final["plot"] = None

In [126]:
# הרצה על כל 5000 הסרטים
for idx, row in df_final.iterrows():
    
    # דילוג על סרטים שכבר יש להם Language ו-Country
    if pd.notna(df_final.at[idx, "Language"]) and pd.notna(df_final.at[idx, "Country"]):
        continue
    
    title = row["primaryTitle"]
    year = row["startYear"]
    
    # Wikipedia
    page = search_wikipedia_page(title, year)
    info = get_wikipedia_movie_info(page)
    
    df_final.at[idx, "Language"] = info["Language"]
    df_final.at[idx, "Country"] = info["Country"]
    df_final.at[idx, "budget"] = info["budget"]
    df_final.at[idx, "BoxOffice"] = info["BoxOffice"]
    
    # plot מ-OMDb
    df_final.at[idx, "plot"] = get_plot_from_omdb(title, year)
    
    time.sleep(5)
    
    if idx % 100 == 0:
        df_final.to_csv("dataset25.csv", index=False, encoding="utf-8-sig")

df_final.to_csv("dataset25.csv", index=False, encoding="utf-8-sig")

200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
No infobox found
200
200
200
200
200
200
200
200
200
200

<div dir="rtl">

#### תיקון והוספת העמודות הרלוונטיות:

In [130]:
df_final = df_final[["tconst", "primaryTitle", "startYear", "genres", "lead_actors_ids", 
                      "runtimeMinutes", "averageRating", "numVotes", 
                      "Language", "Country", "budget", "BoxOffice", "plot"]].copy()

print(list(df_final.columns))

['tconst', 'primaryTitle', 'startYear', 'genres', 'lead_actors_ids', 'runtimeMinutes', 'averageRating', 'numVotes', 'Language', 'Country', 'budget', 'BoxOffice', 'plot']


<div dir="rtl">

#### יצירת קובץ csv לגיבוי:

In [132]:
df_final.to_csv("dataset25.csv", index=False, encoding="utf-8-sig")

<div dir="rtl">

#### הגענו לדאטה שרצינו, נבדוק את הtype של כל עמודה ונשנה לtype הרלוונטי:

In [245]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   tconst           5000 non-null   string 
 1   primaryTitle     5000 non-null   string 
 2   startYear        5000 non-null   Int64  
 3   genres           5000 non-null   object 
 4   lead_actors_ids  5000 non-null   object 
 5   runtimeMinutes   5000 non-null   Int64  
 6   averageRating    5000 non-null   float64
 7   numVotes         5000 non-null   Int64  
 8   Language         3736 non-null   string 
 9   Country          3737 non-null   string 
 10  budget           1350 non-null   float64
 11  BoxOffice        1684 non-null   float64
 12  plot             4745 non-null   string 
dtypes: Int64(3), float64(3), object(2), string(5)
memory usage: 522.6+ KB


In [247]:
import ast

# פונקציה להפיכת טקסט של כסף למספר במיליוני דולרים
def money_to_millions(value):
    if pd.isna(value):
        return np.nan
    
    text = str(value).lower()
    
    # ניקוי סימנים מיותרים
    text = text.replace(",", "")
    text = text.replace("$", "")
    text = text.replace("us", "")
    text = text.replace("usd", "")
    text = text.replace("–", "-")
    text = text.replace("—", "-")
    
    # חילוץ כל המספרים מהטקסט
    numbers = re.findall(r"\d+(?:\.\d+)?", text)
    
    if len(numbers) == 0:
        return np.nan
    
    numbers = [float(num) for num in numbers]
    
    # אם יש טווח, לוקחים ממוצע
    amount = sum(numbers) / len(numbers)
    
    # המרה למיליוני דולרים
    if "billion" in text:
        amount = amount * 1000
    elif "million" in text:
        amount = amount
    else:
        # אם לא כתוב million/billion, נניח שזה בדולרים רגילים ונמיר למיליונים
        amount = amount / 1_000_000
    
    return amount


# פונקציה להפיכת genres לרשימה
def genres_to_list(value):

    # אם זה כבר רשימה
    if isinstance(value, list):
        return value

    # אם חסר ערך
    if value is None:
        return []

    # אם זה NaN
    if isinstance(value, float) and pd.isna(value):
        return []

    return str(value).split(",")


# פונקציה בטוחה ל-lead_actors_ids
def actors_to_list(value):
    if isinstance(value, list):
        return value
    
    if pd.isna(value):
        return []
    
    try:
        return ast.literal_eval(value)
    except:
        return str(value).split(",")


# המרות טיפוסים לפי דרישות המטלה

df_final["tconst"] = df_final["tconst"].astype("string")
df_final["primaryTitle"] = df_final["primaryTitle"].astype("string")

df_final["startYear"] = pd.to_numeric(df_final["startYear"], errors="coerce").astype("Int64")
df_final["runtimeMinutes"] = pd.to_numeric(df_final["runtimeMinutes"], errors="coerce").astype("Int64")
df_final["numVotes"] = pd.to_numeric(df_final["numVotes"], errors="coerce").astype("Int64")

df_final["averageRating"] = pd.to_numeric(df_final["averageRating"], errors="coerce").astype(float)

df_final["genres"] = df_final["genres"].apply(genres_to_list)
df_final["lead_actors_ids"] = df_final["lead_actors_ids"].apply(actors_to_list)

df_final["Language"] = df_final["Language"].astype("string")
df_final["Country"] = df_final["Country"].astype("string")
df_final["plot"] = df_final["plot"].astype("string")

df_final["budget"] = df_final["budget"].apply(money_to_millions).astype(float)
df_final["BoxOffice"] = df_final["BoxOffice"].apply(money_to_millions).astype(float)

df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   tconst           5000 non-null   string 
 1   primaryTitle     5000 non-null   string 
 2   startYear        5000 non-null   Int64  
 3   genres           5000 non-null   object 
 4   lead_actors_ids  5000 non-null   object 
 5   runtimeMinutes   5000 non-null   Int64  
 6   averageRating    5000 non-null   float64
 7   numVotes         5000 non-null   Int64  
 8   Language         3736 non-null   string 
 9   Country          3737 non-null   string 
 10  budget           1350 non-null   float64
 11  BoxOffice        1684 non-null   float64
 12  plot             4745 non-null   string 
dtypes: Int64(3), float64(3), object(2), string(5)
memory usage: 522.6+ KB


<div dir="rtl">

#### נבדוק כמה ערכים חסרים בכל עמודה:

In [250]:
missing = df_final.isnull().sum()
missing_pct = (df_final.isnull().sum() / len(df_final) * 100).round(2)
print(pd.DataFrame({"Missing Values": missing, "Missing Percentage": missing_pct}))

                 Missing Values  Missing Percentage
tconst                        0                0.00
primaryTitle                  0                0.00
startYear                     0                0.00
genres                        0                0.00
lead_actors_ids               0                0.00
runtimeMinutes                0                0.00
averageRating                 0                0.00
numVotes                      0                0.00
Language                   1264               25.28
Country                    1263               25.26
budget                     3650               73.00
BoxOffice                  3316               66.32
plot                        255                5.10


<div dir="rtl">

#### נראה שכל הobjects באמת list/float:

In [159]:
type(df_final.loc[0, "genres"])

list

In [161]:
type(df_final.loc[0, "lead_actors_ids"])

list

In [177]:
df_final[["budget", "BoxOffice"]].dtypes

budget       float64
BoxOffice    float64
dtype: object

<div dir="rtl">

#### נבדוק שאין יותר סימני דולר, טקסט, טווחים:

In [190]:
df_final[
    df_final["budget"].astype(str).str.contains(r"\$|million|billion", na=False)
]

,tconst,primaryTitle,startYear,genres,lead_actors_ids,runtimeMinutes,averageRating,numVotes,Language,Country,budget,BoxOffice,plot


In [192]:
df_final[
    df_final["BoxOffice"].astype(str).str.contains(r"\$|million|billion", na=False)
]

,tconst,primaryTitle,startYear,genres,lead_actors_ids,runtimeMinutes,averageRating,numVotes,Language,Country,budget,BoxOffice,plot


In [194]:
df_final[df_final["primaryTitle"] == "Coco"][["budget", "BoxOffice"]]

,budget,BoxOffice
3,0.0002,0.000814


In [200]:
money_to_millions("$175–225 million")

200.0

<div dir="rtl">

#### נבדוק כפילויות לפי מזהה הסרט:

In [171]:
df_final.duplicated(subset=["tconst"]).sum()

0

<div dir="rtl">

#### נשמור כקובץ csv:

In [262]:
df_final.to_csv("dataset25.csv", index=False)